# Extracción y Diagnóstico de Datos de Establecimientos del MINEDUC

Este notebook automatiza la descarga de registros de establecimientos educativos desde el portal del MINEDUC de Guatemala utilizando Selenium, limpia la información y ejecuta un diagnóstico paso a paso de calidad de datos.

## 1. Importación de Librerías y Configuración Global

In [1]:
import os
import time
import glob
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.action_chains import ActionChains

# Parámetros globales de búsqueda y almacenamiento
BASE_URL = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/"
CARPETA_DESCARGAS = os.path.join(os.getcwd(), "descargas")
CSV_SALIDA = "mineduc_establecimientos_unificado.csv"

DEPARTAMENTOS = [
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA",
    "CIUDAD CAPITAL", "EL PROGRESO", "ESCUINTLA", "GUATEMALA",
    "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA", "PETEN",
    "QUETZALTENANGO", "QUICHE", "RETALHULEU", "SACATEPEQUEZ",
    "SAN MARCOS", "SANTA ROSA", "SOLOLA", "SUCHITEPEQUEZ",
    "TOTONICAPAN", "ZACAPA"
]

NIVELES = ["BASICO", "DIVERSIFICADO"]

## 2. Funciones Auxiliares para Web Scraping

In [2]:
def encontrar_select_por_nombre(driver, nombre_parcial):
    selects = driver.find_elements(By.TAG_NAME, "select")
    for sel in selects:
        name = sel.get_attribute("name")
        if name and nombre_parcial in name:
            return sel
    return None

def encontrar_boton_por_src(driver, src_parcial):
    inputs = driver.find_elements(By.XPATH, "//input[@type='image']")
    for inp in inputs:
        src = inp.get_attribute("src")
        if src and src_parcial in src:
            return inp
    return None

def hacer_scroll_hasta_elemento(driver, elemento):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elemento)
    time.sleep(0.5)

def clic_seguro(driver, elemento):
    try:
        elemento.click()
    except Exception:
        ActionChains(driver).move_to_element(elemento).click().perform()

def esperar_descarga(carpeta, timeout=30):
    archivos_iniciales = set(glob.glob(os.path.join(carpeta, "*.xls")))
    start = time.time()
    while time.time() - start < timeout:
        archivos_actuales = set(glob.glob(os.path.join(carpeta, "*.xls")))
        nuevos = archivos_actuales - archivos_iniciales
        if nuevos:
            return max(nuevos, key=os.path.getctime)
        time.sleep(1)
    raise TimeoutError("No se detectó la descarga del archivo .xls")

def leer_tabla_desde_archivo_xls(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='latin1') as f:
        contenido = f.read()
    tablas = pd.read_html(contenido)
    if not tablas:
        raise ValueError("No se encontraron tablas en el archivo")
    df = max(tablas, key=lambda t: t.shape[0])
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    df = df.dropna(how='all')
    return df

## 3. Extracción y Carga Inicial de Datos

In [3]:
def descargar_departamento_nivel(driver, departamento, nivel):
    driver.get(BASE_URL)
    wait = WebDriverWait(driver, 15)

    select_depto = encontrar_select_por_nombre(driver, "cmbDepartamento")
    if not select_depto:
        raise Exception("No se encontró el select de Departamento")
    Select(select_depto).select_by_visible_text(departamento)
    time.sleep(1.5)

    select_nivel = encontrar_select_por_nombre(driver, "cmbNivel")
    if not select_nivel:
        raise Exception("No se encontró el select de Nivel")
    Select(select_nivel).select_by_visible_text(nivel)
    time.sleep(0.5)

    boton_buscar = driver.find_element(By.XPATH, "//input[@type='image' and contains(@src, 'seleccionar_estab.gif')]")
    hacer_scroll_hasta_elemento(driver, boton_buscar)
    clic_seguro(driver, boton_buscar)
    time.sleep(4)

    try:
        wait.until(EC.presence_of_element_located((By.XPATH, "//table")))
    except TimeoutException:
        print(f"  Sin resultados para {departamento} - {nivel}")
        return None

    boton_excel = encontrar_boton_por_src(driver, "export_excel.gif")
    if not boton_excel:
        try:
            boton_excel = driver.find_element(By.XPATH, "//*[contains(text(),'Generar Archivo de Excel')]")
        except Exception:
            pass
    if not boton_excel:
        raise Exception("No se encontró el botón de Excel")

    hacer_scroll_hasta_elemento(driver, boton_excel)
    wait.until(EC.element_to_be_clickable(boton_excel))
    boton_excel.click()

    ruta_archivo = esperar_descarga(CARPETA_DESCARGAS, timeout=30)
    nuevo_nombre = f"{departamento}_{nivel}.xls"
    nueva_ruta = os.path.join(CARPETA_DESCARGAS, nuevo_nombre)
    if os.path.exists(nueva_ruta):
        base, ext = os.path.splitext(nuevo_nombre)
        nueva_ruta = os.path.join(CARPETA_DESCARGAS, f"{base}_{int(time.time())}{ext}")
    os.rename(ruta_archivo, nueva_ruta)
    print(f"  Descargado: {os.path.basename(nueva_ruta)}")
    return nueva_ruta

def descargar_y_unificar():
    os.makedirs(CARPETA_DESCARGAS, exist_ok=True)

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    prefs = {
        "download.default_directory": CARPETA_DESCARGAS,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)
    archivos_descargados = []

    try:
        print("Iniciando descarga de todos los departamentos (Básico y Diversificado)...")
        for depto in DEPARTAMENTOS:
            for nivel in NIVELES:
                try:
                    print(f"Procesando {depto} - {nivel}...")
                    ruta = descargar_departamento_nivel(driver, depto, nivel)
                    if ruta:
                        archivos_descargados.append(ruta)
                except Exception as e:
                    print(f"  ERROR: {e}")
                time.sleep(2)

        if not archivos_descargados:
            raise SystemExit("No se descargó ningún archivo.")

        dataframes = []
        for archivo in archivos_descargados:
            try:
                df = leer_tabla_desde_archivo_xls(archivo)
                nombre_base = os.path.basename(archivo).replace(".xls", "")
                partes = nombre_base.split("_")
                if len(partes) >= 2:
                    df["DEPARTAMENTO_CONSULTADO"] = partes[0]
                    df["NIVEL_CONSULTADO"] = partes[1]
                dataframes.append(df)
            except Exception as e:
                print(f"Error al leer {archivo}: {e}")

        df_unificado = pd.concat(dataframes, ignore_index=True)
        df_unificado.columns = [str(c).strip().upper() for c in df_unificado.columns]

        if "CODIGO" in df_unificado.columns:
            df_unificado = df_unificado[df_unificado["CODIGO"].notna()]
            df_unificado = df_unificado[df_unificado["CODIGO"].astype(str).str.strip() != ""]

        categoricas = ["DEPARTAMENTO", "MUNICIPIO", "NIVEL", "SECTOR", "AREA",
                       "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL"]
        for col in categoricas:
            if col in df_unificado.columns:
                df_unificado[col] = df_unificado[col].astype(str).str.upper().str.strip()
                df_unificado[col] = df_unificado[col].replace({"NAN": pd.NA, "NONE": pd.NA, "---": pd.NA})

        if "TELEFONO" in df_unificado.columns:
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].astype(str).str.replace(r"\.0$", "", regex=True)
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].replace({"nan": pd.NA, "<NA>": pd.NA})

        if "CODIGO" in df_unificado.columns:
            df_unificado["_no_nulos"] = df_unificado.notna().sum(axis=1)
            df_unificado = df_unificado.sort_values("_no_nulos", ascending=False).drop_duplicates(subset=["CODIGO"]).drop(columns="_no_nulos")

        df_unificado.to_csv(CSV_SALIDA, index=False, encoding="utf-8-sig")
        return df_unificado
    finally:
        driver.quit()

# Obtención del Dataset
if os.path.exists(CSV_SALIDA):
    print(f"Cargando archivo existente '{CSV_SALIDA}'...")
    df = pd.read_csv(CSV_SALIDA, encoding='utf-8-sig')
else:
    print("Descargando datos...")
    df = descargar_y_unificar()

Cargando archivo existente 'mineduc_establecimientos_unificado.csv'...


## 4. Diagnóstico de Calidad de Datos

### a. Número de registros y variables
**Descripción:** Evalúa la dimensión total del conjunto de datos identificando la cantidad total de filas (registros) y columnas (variables).

In [4]:
print("a. Número de registros y variables:")
print(f"   - Registros (filas): {df.shape[0]}")
print(f"   - Variables (columnas): {df.shape[1]}")

a. Número de registros y variables:
   - Registros (filas): 27413
   - Variables (columnas): 19


### b. Tipo de dato de cada variable
**Descripción:** Detalla el tipo de dato asignado a cada columna (e.g., `object`, `int64`, `float64`) para asegurar su correcta interpretación analítica.

In [5]:
print("b. Tipo de dato de cada variable:")
for col in df.columns:
    print(f"   - {col}: {df[col].dtype}")

b. Tipo de dato de cada variable:
   - CODIGO: object
   - DISTRITO: object
   - DEPARTAMENTO: object
   - MUNICIPIO: object
   - ESTABLECIMIENTO: object
   - DIRECCION: object
   - TELEFONO: object
   - SUPERVISOR: object
   - DIRECTOR: object
   - NIVEL: object
   - SECTOR: object
   - AREA: object
   - STATUS: object
   - MODALIDAD: object
   - JORNADA: object
   - PLAN: object
   - DEPARTAMENTAL: object
   - DEPARTAMENTO_CONSULTADO: object
   - NIVEL_CONSULTADO: object


### c. Cantidad y porcentaje de valores faltantes por variable
**Descripción:** Cuantifica la presencia de valores nulos o ausentes en cada variable, calculando su proporción respecto al total de registros.

In [6]:
print("c. Cantidad y porcentaje de valores faltantes por variable:")
faltantes = df.isna().sum()
total = len(df)
for col in df.columns:
    nulos = faltantes[col]
    pct = (nulos / total) * 100
    print(f"   - {col}: {nulos} ({pct:.1f}%)")

c. Cantidad y porcentaje de valores faltantes por variable:
   - CODIGO: 0 (0.0%)
   - DISTRITO: 1616 (5.9%)
   - DEPARTAMENTO: 0 (0.0%)
   - MUNICIPIO: 0 (0.0%)
   - ESTABLECIMIENTO: 7 (0.0%)
   - DIRECCION: 279 (1.0%)
   - TELEFONO: 2800 (10.2%)
   - SUPERVISOR: 1620 (5.9%)
   - DIRECTOR: 4634 (16.9%)
   - NIVEL: 0 (0.0%)
   - SECTOR: 0 (0.0%)
   - AREA: 0 (0.0%)
   - STATUS: 0 (0.0%)
   - MODALIDAD: 0 (0.0%)
   - JORNADA: 0 (0.0%)
   - PLAN: 0 (0.0%)
   - DEPARTAMENTAL: 0 (0.0%)
   - DEPARTAMENTO_CONSULTADO: 0 (0.0%)
   - NIVEL_CONSULTADO: 0 (0.0%)


### d. Cantidad de valores únicos por variable
**Descripción:** Determina la cardinalidad de cada columna mostrando cuántos valores distintivos o categorías contiene.

In [7]:
print("d. Cantidad de valores únicos por variable:")
for col in df.columns:
    print(f"   - {col}: {df[col].nunique()}")

d. Cantidad de valores únicos por variable:
   - CODIGO: 27413
   - DISTRITO: 2017
   - DEPARTAMENTO: 23
   - MUNICIPIO: 357
   - ESTABLECIMIENTO: 10607
   - DIRECCION: 14724
   - TELEFONO: 13278
   - SUPERVISOR: 1501
   - DIRECTOR: 11459
   - NIVEL: 2
   - SECTOR: 4
   - AREA: 3
   - STATUS: 5
   - MODALIDAD: 2
   - JORNADA: 6
   - PLAN: 13
   - DEPARTAMENTAL: 26
   - DEPARTAMENTO_CONSULTADO: 23
   - NIVEL_CONSULTADO: 2


### e. Registros duplicados exactos
**Descripción:** Cuenta las filas que son idénticas en todas sus columnas para detectar redundancia total en la información.

In [8]:
duplicados_exactos = df.duplicated().sum()
print(f"e. Registros duplicados exactos: {duplicados_exactos}")

e. Registros duplicados exactos: 0


### f. Variables con valores fuera de dominio o inconsistentes
**Descripción:** Contrasta las opciones existentes en variables categóricas clave contra un conjunto predefinido de valores válidos esperados.

In [9]:
print("f. Variables con valores fuera de dominio o inconsistentes:")
dominios_esperados = {
    "DEPARTAMENTO": sorted(set(DEPARTAMENTOS) | {"ALTA VERAPAZ", "BAJA VERAPAZ", "CIUDAD CAPITAL"}),
    "NIVEL": ["BASICO", "DIVERSIFICADO", "PRIMARIA", "PARVULOS", "PREPRIMARIA BILINGUE", "PRIMARIA DE ADULTOS"],
    "SECTOR": ["OFICIAL", "PRIVADO", "MUNICIPAL", "COOPERATIVA"],
    "AREA": ["URBANA", "RURAL"],
    "STATUS": ["ABIERTA", "CERRADA DEFINITIVAMENTE", "CERRADA TEMPORALMENTE", "TEMPORAL NOMBRAMIENTO"],
    "MODALIDAD": ["MONOLINGUE", "BILINGUE"],
    "JORNADA": ["MATUTINA", "VESPERTINA", "NOCTURNA", "DOBLE", "SIN JORNADA"],
    "PLAN": ["DIARIO(REGULAR)", "SABATINO", "DOMINICAL", "FIN DE SEMANA", "IRREGULAR", "MIXTO", "A DISTANCIA", "VIRTUAL A DISTANCIA", "SEMIPRESENCIAL (FIN DE SEMANA)", "SEMIPRESENCIAL (UN DÍA A LA SEMANA)"]
}
for col, esperados in dominios_esperados.items():
    if col in df.columns:
        valores_reales = set(df[col].dropna().unique())
        fuera = valores_reales - set(esperados)
        if fuera:
            print(f"   - {col}: valores fuera del dominio esperado: {fuera}")
        else:
            print(f"   - {col}: todos los valores están dentro del dominio esperado.")

f. Variables con valores fuera de dominio o inconsistentes:
   - DEPARTAMENTO: todos los valores están dentro del dominio esperado.
   - NIVEL: todos los valores están dentro del dominio esperado.
   - SECTOR: todos los valores están dentro del dominio esperado.
   - AREA: valores fuera del dominio esperado: {'SIN ESPECIFICAR'}
   - STATUS: valores fuera del dominio esperado: {'TEMPORAL TITULOS'}
   - MODALIDAD: todos los valores están dentro del dominio esperado.
   - JORNADA: valores fuera del dominio esperado: {'INTERMEDIA'}
   - PLAN: valores fuera del dominio esperado: {'SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)', 'INTERCALADO', 'SEMIPRESENCIAL'}


### g. Variables con formatos inconsistentes
**Descripción:** Inspecciona la sintaxis de campos específicos (e.g., código institucional y teléfonos) para validar que cumplan con la estructura estandarizada.

In [10]:
print("g. Variables con formatos inconsistentes:")

# Evaluación del patrón de CODIGO
inconsistentes = []
if "CODIGO" in df.columns:
    codigos = df["CODIGO"].dropna().astype(str)
    patron = re.compile(r'^\d{2}-\d{2}-\d{4}-\d{2}$')
    inconsistentes = codigos[~codigos.str.match(patron)]
    if len(inconsistentes) > 0:
        print(f"   - CODIGO: {len(inconsistentes)} registros no siguen el formato ##-##-####-##")
        print(f"     Ejemplos: {inconsistentes.head(3).tolist()}")
    else:
        print("   - CODIGO: todos los códigos siguen el formato esperado.")

# Evaluación de TELEFONO
no_numericos = []
if "TELEFONO" in df.columns:
    telefonos = df["TELEFONO"].dropna().astype(str)
    limpios = telefonos.str.replace(r'[\s\-\(\)]', '', regex=True)
    no_numericos = limpios[~limpios.str.isdigit()]
    if len(no_numericos) > 0:
        print(f"   - TELEFONO: {len(no_numericos)} registros contienen caracteres no numéricos.")
        print(f"     Ejemplos: {no_numericos.head(3).tolist()}")
    else:
        print("   - TELEFONO: todos los valores son numéricos (tras limpieza).")

g. Variables con formatos inconsistentes:
   - CODIGO: todos los códigos siguen el formato esperado.
   - TELEFONO: 99 registros contienen caracteres no numéricos.
     Ejemplos: ['55074625,58722678', '22206556Y22516346', '22534814y22381691']


### h. Identificación de problemas potenciales de calidad de datos
**Descripción:** Sintetiza los hallazgos principales de las pruebas previas y consolida los riesgos potenciales encontrados en la base de datos.

In [11]:
print("h. Identificación de problemas potenciales de calidad de datos:")
problemas = []

altos_faltantes = [col for col in df.columns if (df[col].isna().sum()/len(df))*100 > 50]
if altos_faltantes:
    problemas.append(f"   - Columnas con más del 50% de datos faltantes: {altos_faltantes}")
if duplicados_exactos > 0:
    problemas.append(f"   - Existen {duplicados_exactos} filas duplicadas exactas.")
if any(fuera for col, fuera in dominios_esperados.items() if col in df.columns):
    problemas.append("   - Algunas variables categóricas tienen valores fuera del dominio esperado (ver sección f).")
if "CODIGO" in df.columns and len(inconsistentes) > 0:
    problemas.append(f"   - {len(inconsistentes)} códigos tienen formato inesperado.")
if "TELEFONO" in df.columns and len(no_numericos) > 0:
    problemas.append(f"   - {len(no_numericos)} teléfonos contienen caracteres no numéricos.")

if problemas:
    for p in problemas:
        print(p)
else:
    print("   - No se detectaron problemas graves de calidad de datos.")

h. Identificación de problemas potenciales de calidad de datos:
   - Algunas variables categóricas tienen valores fuera del dominio esperado (ver sección f).
   - 99 teléfonos contienen caracteres no numéricos.


## 5. Plan de Limpieza

### 5.1 Resumen

Dataset crudo: 27,413 filas, 19 variables, todas tipo object. 0 duplicados exactos. Problemas principales: NA reales y disfrazados (".", "-", "SIN DATO"), formatos inconsistentes en TELEFONO y DISTRITO, categorías que el diagnóstico marcó como "fuera de dominio" pero en realidad son válidas (SIN ESPECIFICAR, INTERMEDIA, etc.), 2 columnas redundantes (DEPARTAMENTO_CONSULTADO, NIVEL_CONSULTADO), y posibles duplicados parciales en ESTABLECIMIENTO (mismo plantel con BASICO/DIVERSIFICADO bajo distinto CODIGO).

Principio general: no se tocan tildes ni ortografía de nombres/direcciones; solo espacios, placeholders y puntuación mal usada.

### 5.2 Variables prioritarias

TELEFONO, DIRECTOR, ESTABLECIMIENTO, DIRECCION y DISTRITO son las que más limpieza requieren (ver detalle en 5.3). MUNICIPIO necesita validación contra catálogo; AREA/STATUS/JORNADA/PLAN solo necesitan ampliar el dominio de referencia; DEPARTAMENTO_CONSULTADO y NIVEL_CONSULTADO se eliminarán por ser redundantes.

### 5.3 División del trabajo

Se dividen las 19 columnas entre los 3 integrantes del grupo. Carga desigual a propósito: Olivier ya avanzó más trabajo antes (scraping y unificación de los datos), así que recibe menos columnas.

| Integrante | Columnas asignadas | Cantidad |
|---|---|---|
| Erick | ESTABLECIMIENTO, DIRECCION, TELEFONO, SUPERVISOR, DIRECTOR, NIVEL, DISTRITO, MUNICIPIO | 8 |
| Fabián | SECTOR, AREA, STATUS, MODALIDAD, JORNADA, PLAN, DEPARTAMENTAL | 7 |
| Olivier | CODIGO, DEPARTAMENTO, DEPARTAMENTO_CONSULTADO, NIVEL_CONSULTADO | 4 |

Cada quien documenta sus transformaciones en la tabla de registro de transformaciones (sección 6).

### 5.4 Estrategia por variable

| Variable | Problema | Regla | Riesgo |
|---|---|---|---|
| CODIGO | Ninguno; formato 100% válido | Mantener como texto; validar con regex | Perder ceros a la izquierda si se numeriza |
| DISTRITO | Formato inconsistente (3/6/10 caracteres) | strip(); documentar variantes; NA si falta | Inventar un código al forzar un formato único |
| DEPARTAMENTO | Ninguno | strip(); convertir a category | Confundir CIUDAD CAPITAL con GUATEMALA |
| MUNICIPIO | "ZONA N" en Ciudad Capital en vez de municipio | Validar contra catálogo INE; marcar zonas con bandera | Perder granularidad si se fuerza a "GUATEMALA" |
| DEPARTAMENTAL | Un departamento puede tener varias oficinas | strip(); category; documentar diferencia con DEPARTAMENTO | Confundirla con el departamento geográfico |
| DEPARTAMENTO_CONSULTADO / NIVEL_CONSULTADO | 100% redundantes | Eliminar | Perder trazabilidad de la extracción (bajo) |
| ESTABLECIMIENTO | Puntuación mal usada (comilla invertida, guion bajo); posibles duplicados parciales | strip(); corregir puntuación solo en casos puntuales; fuzzy-match para detectar duplicados sin eliminar | Dañar ortografía maya o fusionar escuelas distintas |
| DIRECCION | 1% NA + placeholders; 26 registros no en mayúsculas | Placeholders exactos a NA; homologar mayúsculas; strip() | Borrar un "-" que sea parte real de la dirección |
| SUPERVISOR | 5.9% NA | strip(); similitud solo para detectar errores de tecleo | Fusionar dos personas distintas por error |
| DIRECTOR | 16.9% NA + 95 placeholders | Placeholders exactos a NA; strip() | Igual que SUPERVISOR |
| TELEFONO | Formatos dispares; 84 celdas con más de un número | Derivar TELEFONO_PRINCIPAL/LISTA; mantener como texto | Separador no detectado deja el caso sin dividir |
| NIVEL / NIVEL_CONSULTADO | Ninguno | strip(); category | — |
| SECTOR | Ninguno | strip(); category | — |
| AREA | "SIN ESPECIFICAR" fuera del dominio inicial del diagnóstico | Agregar como categoría válida | Confundirla con NA |
| STATUS | "TEMPORAL TITULOS" fuera del dominio inicial | Agregar como categoría válida | Confundir con TEMPORAL NOMBRAMIENTO |
| MODALIDAD | Ninguno | strip(); category | — |
| JORNADA | "INTERMEDIA" fuera del dominio inicial | Agregar como categoría válida | Confundir con DOBLE |
| PLAN | 3 categorías fuera del dominio inicial | Ampliar catálogo a las 13 categorías reales | Fusionar por error variantes de SEMIPRESENCIAL